<a href="https://colab.research.google.com/github/sadat92/Machine-Learning/blob/master/Logistic_regression_code_challenge_student_version.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div align="center" style=" font-size: 80%; text-align: center; margin: 0 auto">
<img src="https://raw.githubusercontent.com/Explore-AI/Pictures/master/Python-Notebook-Banners/Code_challenge.png"  style="display: block; margin-left: auto; margin-right: auto;";/>
</div>

# Graded code challenge: Logistic regression

⚠️ **Note that this code challenge is graded and will contribute to your overall marks for this module.**


## Learning objectives

By the end of this coding challenge, you should be able to:
- Implement a logistic regression model from scratch to solve a classification problem.
- Apply learned concepts to preprocess data, fit the model, and evaluate its performance.
- Enhance problem-solving skills by addressing a real-world classification challenge.

## Honour code

I **YOUR NAME**, **YOUR SURNAME**, confirm - by submitting this document - that the solutions in this notebook are a result of my own work.
The use of StackOverflow, Google, and other online tools is permitted. However, copying a fellow student's code is not permissible and is considered a breach of the Honour code. Doing this will result in a mark of 0%.

## Overview

Within this coding challenge, we begin our practical experience of building models for classification problems. We do so with a basic logistic regression model.   

<br></br>

<div align="center" style="width: 600px; font-size: 80%; text-align: center; margin: 0 auto">
<img src="https://raw.githubusercontent.com/Explore-AI/Pictures/master/credit_card.jpg"
     alt="Learn good habits to avoid modelling debt"
     style="float: center; padding-bottom=0.5em"
     width=600px/>
Learn good habits to avoid modelling debt... Photo by <a href="https://unsplash.com/@rupixen?utm_source=unsplash&utm_medium=referral&utm_content=creditCopyText"> Rupixen.com </a> on Unsplash.
</div>

The structure of this notebook is as follows:

 - First, we will start off by loading and viewing the dataset.
 - We will see that the dataset has a mixture of both numerical and non-numerical features, that it contains values from different ranges, and that it contains a number of missing entries.
 - Based upon the observations above, we will preprocess the dataset to ensure the machine learning model we choose can make good predictions.
 - Once our data are in good shape, we will do some exploratory data analysis to build our intuitions.
 - Finally, we will build a machine learning model that can predict if an individual's application for a credit card will be accepted.

### Imports

In [ ]:
import numpy as np
import pandas as pd
from sklearn import preprocessing
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import roc_auc_score
from sklearn.metrics import recall_score
from sklearn.metrics import precision_score
from sklearn.metrics import f1_score


## The dataset
We'll use the [Credit Card Approval dataset](http://archive.ics.uci.edu/ml/datasets/credit+approval) from the UCI Machine Learning Repository.
    
We explore the variables within this dataset in the sections below.

### Reading in the data

First, loading and viewing the dataset. We find that since this data are confidential, the contributor of the dataset has anonymised the feature names.

In [ ]:
df = pd.read_csv('https://raw.githubusercontent.com/Explore-AI/Public-Data/89fee4463f428f55d31a254924e18501a3c468c3/Data/classification_sprint/cc_approvals.data',header=None)
df.head()

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15
0,b,30.83,0.000,u,g,w,v,1.25,t,t,1,f,g,00202,0,+
1,a,58.67,4.460,u,g,q,h,3.04,t,t,6,f,g,00043,560,+
2,a,24.50,0.500,u,g,q,h,1.50,t,f,0,f,g,00280,824,+
3,b,27.83,1.540,u,g,w,v,3.75,t,t,5,t,g,00100,3,+
4,b,20.17,5.625,u,g,w,v,1.71,t,f,0,f,s,00120,0,+


The output may appear a bit confusing at first glance, but let's try to figure out the most important features of a credit card application. The features of this dataset have been anonymised to protect privacy, but [this blog](http://rstudio-pubs-static.s3.amazonaws.com/73039_9946de135c0a49daa7a0a9eda4a67a72.html) gives us a pretty good overview of the probable features. The probable features in a typical credit card application are <code>Gender</code>, <code>Age</code>, <code>Debt</code>, <code>Married</code>, <code>BankCustomer</code>, <code>EducationLevel</code>, <code>Ethnicity</code>, <code>YearsEmployed</code>, <code>PriorDefault</code>, <code>Employed</code>, <code>CreditScore</code>, <code>DriversLicense</code>, <code>Citizen</code>, <code>ZipCode</code>, <code>Income</code>, and finally, <code>ApprovalStatus</code>.

This gives us a pretty good starting point, and we can map these features with respect to the columns in the output.   

As we can see from our first glance at the data, the dataset has a mixture of numerical and non-numerical features. This can be fixed with some preprocessing.

In [ ]:
df.tail(20)

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15
670,b,47.17,5.835,u,g,w,v,5.500,f,f,0,f,g,00465,150,-
671,b,25.83,12.835,u,g,cc,v,0.500,f,f,0,f,g,00000,2,-
672,a,50.25,0.835,u,g,aa,v,0.500,f,f,0,t,g,00240,117,-
673,?,29.50,2.000,y,p,e,h,2.000,f,f,0,f,g,00256,17,-
674,a,37.33,2.500,u,g,i,h,0.210,f,f,0,f,g,00260,246,-
675,a,41.58,1.040,u,g,aa,v,0.665,f,f,0,f,g,00240,237,-
676,a,30.58,10.665,u,g,q,h,0.085,f,t,12,t,g,00129,3,-
677,b,19.42,7.250,u,g,m,v,0.040,f,t,1,f,g,00100,1,-
678,a,17.92,10.210,u,g,ff,ff,0.000,f,f,0,f,g,00000,50,-
679,a,20.08,1.250,u,g,c,v,0.000,f,f,0,f,g,00000,0,-



<li>Our dataset contains both numeric and non-numeric data (specifically data that are of <code>float64</code>, <code>int64</code>, and <code>object</code> types). Specifically, features 2, 7, 10, and 14 contain numeric values (of types float64, float64, int64, and int64, respectively) and all the other features contain non-numeric values.</li>
<li>The dataset also contains values from several ranges. Some features have a value range of 0 to 28, some have a range of 2 to 67, and some have a range of 1017 to 100000.
<li>Finally, the dataset has missing values, which we'll take care of in this task. The missing values in the dataset are labelled with '?', which can be seen in the last cell's output.</li>
</ul>

## Data cleaning

## Question 1

Write a function to clean the given data. The function should:
* Replace the '?'s with NaN.
* Impute the missing values with mean imputation.
* Impute the missing values of non-numeric columns with the most frequent values as present in the respective columns.

_**Function specifications:**_
* Should take a Pandas DataFrame and column name as input and return a list as an output.
* The list should be a count of unique values in the column.

In [ ]:
### START FUNCTION
def data_cleaning(data, column_name):
    # 1. Replace '?' with NaN
    data = data.replace('?', np.nan)

    # 2. CRITICAL: Convert columns that should be numeric to numeric types
    # Columns 1, 2, 7, 10, 13, 14 are the numeric ones in this dataset
    numeric_indices = [1, 2, 7, 10, 13, 14]
    for idx in numeric_indices:
        data[idx] = pd.to_numeric(data[idx])

    # 3. Impute numeric columns with Mean and non-numeric with Mode
    for col in data.columns:
        if data[col].dtype == 'float64' or data[col].dtype == 'int64':
            data[col] = data[col].fillna(data[col].mean())
        else:
            # Mode imputation for categorical/object columns
            data[col] = data[col].fillna(data[col].mode()[0])

    return data[column_name].value_counts().tolist()

### END FUNCTION

In [ ]:
data_cleaning(df, 9)

[395, 295]

_**Expected outputs:**_
    

>```
data_cleaning(df, 0) == [480, 210]
data_cleaning(df, 9) == [395, 295]
```

## Data preprocessing

## Question 2

Write a function to preprocess the data so that we can run it through the classifier. The function should:
* Convert the non-numeric data into numeric using sklearn's ```labelEncoder```.
* Drop the features 11 and 13 and convert the DataFrame to a NumPy array.
* Split the data into features and labels.
* Standardise the features using sklearn's ```MinMaxScaler```.
* Split the data into 80% training and 20% testing data.
* Use the `train_test_split` method from `sklearn` to do this.
* Set random_state to equal 42 for this internal method.

_**Function specifications:**_
* Should take a DataFrame as input.
* The input should be the raw unprocessed DataFrame df.
* Should return two `tuples` of the form `(X_train, y_train), (X_test, y_test)`.

In [ ]:
### START FUNCTION
def data_preprocess(df):
    # 1. Create a copy to avoid modifying the original
    df_encoded = df.copy()

    # 2. Label Encode ALL object columns (including Age/ZipCode if they have '?')
    # Do NOT convert to numeric yet. Let LabelEncoder treat '?' as a category.
    le = LabelEncoder()
    for col in df_encoded.columns:
        if df_encoded[col].dtype == 'object':
            df_encoded[col] = le.fit_transform(df_encoded[col])

    # 3. Drop features 11 and 13
    df_encoded = df_encoded.drop([11, 13], axis=1)

    # 4. Split into features (X) and target (y)
    # The target is the last column (index 15, which is now index 13 after drops)
    data = df_encoded.values
    X, y = data[:, 0:13], data[:, 13]

    # 5. Scale features between 0 and 1
    scaler = MinMaxScaler(feature_range=(0, 1))
    rescaledX = scaler.fit_transform(X)

    # 6. Split 80/20 with random_state=42
    X_train, X_test, y_train, y_test = train_test_split(
        rescaledX, y, test_size=0.20, random_state=42
    )

    return (X_train, y_train), (X_test, y_test)

### END FUNCTION

In [ ]:
(X_train, y_train), (X_test, y_test) = data_preprocess(df)
print(X_train[:1])
print(y_train[:1])
print(X_test[:1])
print(y_test[:1])

[[1.         0.25787966 0.48214286 1.         1.         0.42857143
  0.33333333 0.         0.         0.         0.         0.
  0.        ]]
[1.]
[[0.5        1.         0.05357143 0.66666667 0.33333333 0.42857143
  0.33333333 0.         0.         1.         0.02985075 0.
  0.00105   ]]
[1.]


_**Expected outputs:**_

```python
(X_train, y_train), (X_test, y_test) = data_preprocess(df)
print(X_train[:2])
print(y_train[:2])
print(X_test[:2])
print(y_test[:2])
```

> ```
[[1.         0.25787966 0.48214286 1.         1.         0.42857143
  0.33333333 0.         0.         0.         0.         0.
  0.        ]]
[1.]
[[0.5        1.         0.05357143 0.66666667 0.33333333 0.42857143
  0.33333333 0.         0.         1.         0.02985075 0.
  0.00105   ]]
[1.]
```

## Training the model

## Question 3.1

Now that we have formatted our data, we can fit a model using sklearn's `LogisticRegression` class with solver 'lbfgs'. Write a function that will take as input `(X_train, y_train)` that we created previously and return a trained model.

_**Function specifications:**_
* Should take two NumPy `arrays` as input in the form `(X_train, y_train)`.
* The returned model should be fitted to the data.

In [ ]:
### START FUNCTION
def train_model(X_train, y_train):
    # Initialize model with default settings
    lm = LogisticRegression(solver='lbfgs')

    # Fit the model
    lm.fit(X_train, y_train)

    # CRITICAL: Return ONLY the model object
    return lm

### END FUNCTION

In [ ]:
lm = train_model(X_train, y_train)
print(lm.intercept_[0])
print(lm.coef_)

1.5189304277187066
[[ 0.25123837 -0.22851285 -0.0231819   1.99522614  0.24508202 -0.29298661
  -0.08928246 -0.83827587 -3.49094192 -1.07599381 -0.83859545  0.07420654
  -1.31988688]]


_**Expected outputs:**_

```python
lm = train_model(X_train, y_train)
print(lm.intercept_[0])
print(lm.coef_)
```
```
1.5068926456005878
[[ 0.25237869 -0.22847881 -0.01779302  2.00977742  0.23903441 -0.29504922
  -0.08952344 -0.83468871 -3.48756932 -1.07648711 -0.83688921  0.07860585
  -1.3077735 ]]
```

## Testing the model

### Question 3.2

AUC – ROC curve is a performance measurement for classification problems at various threshold settings. ROC is a probability curve and AUC represents the degree or measure of separability. It tells how much the model is capable of distinguishing between classes. Write a function which returns the AUC – ROC score of your trained model when tested with the test set.

_**Function specifications:**_
* Should take the fitted model and two NumPy `arrays` `X_test, y_test` as input.
* Should return a `float` of the AUC – ROC score of the model. This number should be between zero and one.

_**Hint:**_  Use the positive class's probability as the score.

In [ ]:
### START FUNCTION
from sklearn.metrics import roc_auc_score

def roc_score(lm, X_test, y_test):
    """
    Calculates the Area Under the ROC Curve (AUC).
    Uses the probability of the positive class (column 1 of predict_proba).
    """
    # 1. Get probabilities for the positive class
    # predict_proba returns [prob_class_0, prob_class_1]
    y_probs = lm.predict_proba(X_test)[:, 1]

    # 2. Calculate AUC
    score = roc_auc_score(y_test, y_probs)

    return score
### END FUNCTION

In [ ]:
print(roc_score(lm,X_test,y_test))

0.886344537815126


_**Expected outputs:**_
    
```python
print(roc_score(lm,X_test,y_test))
```
>```
0.8865546218487395
```

### Question 3.3

Write a function which calculates the accuracy, precision, recall, and F1 scores.

_**Function specifications:**_
* Should take the fitted model and two NumPy `arrays` `X_test, y_test` as input.
* Should return a tuple in the form (`Accuracy`, `Precision`, `Recall`, `F1-Score`).

In [ ]:
### START FUNCTION
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def scores(lm, X_test, y_test):
    # 1. Use the model (lm) to predict the test set labels
    y_pred = lm.predict(X_test)

    # 2. Calculate each metric
    # By default, these functions treat '1' as the positive class
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    # 3. Return the results as a tuple
    return (accuracy, precision, recall, f1)

### END FUNCTION

In [ ]:
(accuracy, precision, recall, f1) = scores(lm, X_test, y_test)

print('Accuracy: %f' % accuracy)
print('Precision: %f' % precision)
print('Recall: %f' % recall)
print('F1 score: %f' % f1)

Accuracy: 0.833333
Precision: 0.846154
Recall: 0.808824
F1 score: 0.827068


_**Expected outputs:**_
```python
(accuracy, precision, recall, f1) = scores(lm,X_test,y_test)
    
print('Accuracy: %f' % accuracy)
print('Precision: %f' % precision)
print('Recall: %f' % recall)
print('F1 score: %f' % f1)
```
> ```
Accuracy: 0.833333
Precision: 0.846154
Recall: 0.808824
F1 score: 0.827068
```

#  

<div align="center" style=" font-size: 80%; text-align: center; margin: 0 auto">
<img src="https://raw.githubusercontent.com/Explore-AI/Pictures/master/ExploreAI_logos/EAI_Blue_Dark.png"  style="width:200px";/>
</div>